# Déplacer une boucle correcte et préciser ce que la mémoire mesure

CPU, CUDA et précision mixte après la compréhension du calcul.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#cuda-amp-memoire).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Un GPU n’est pas nécessaire pour comprendre la boucle. Cette fiche devient utile quand la taille du modèle ou des lots justifie de mesurer un autre protocole de calcul.

Nous conservons les trois extraits fictifs et les poids connus. CUDA désigne ici le calcul sur un GPU compatible. Le programme choisit CUDA si PyTorch le rend disponible, sinon CPU.

to(device) renvoie l’objet placé à cet endroit : affectez le résultat pour les tenseurs. Changer de device ne change ni les axes ni les annotations. Les cibles de classe restent des entiers long.

AMP utilise plusieurs formats numériques selon les opérations. autocast encadre la passe avant ; le scaler aide à protéger les petits gradients de la branche float16. Les opérations qui demandent davantage de précision peuvent rester en float32.

scale met la perte à l’échelle avant backward. step retire cette échelle et peut éviter une mise à jour si les gradients sont non finis. update ajuste l’échelle. Ici, tout ce mécanisme est désactivé sur CPU : le calcul ordinaire reste exécutable.

| Valeurs | Nombre | Octets en float32 |
| --- | --- | --- |
| Poids | 2×2=4 | 16 |
| Biais | 2 | 8 |
| Paramètres seuls | 6 | 24 |

numel compte les valeurs et element_size leurs octets. Ces 24 octets ne comprennent ni gradients, ni activations, ni états d’optimiseur, ni moteur d’exécution.

Sur CUDA, memory_allocated compte les allocations suivies par PyTorch, pas toute la mémoire du processus ni celle réservée par son cache. Le chemin CPU est vérifié ; la branche CUDA n’a pas été exécutée sur la machine de contrôle. Aucun gain de vitesse ou de mémoire GPU n’est déduit ici.

Si vous combinez AMP et clipping, appelez scaler.unscale_(optimiseur) avant clip_grad_norm_, puis scaler.step. On limite ainsi les gradients dans leur échelle réelle.

## Démonstration 1 · Faire le même pas selon le matériel disponible

La branche CPU constitue le contrôle de référence.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch
from torch import nn


### Calculer l’état suivant

Le matériel disponible est choisi ; les poids initiaux restent imposés.

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modele = nn.Linear(2, 2)


In [5]:
observer('device', device)

### Appliquer le contexte explicite

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [6]:
with torch.no_grad():
    modele.weight.copy_(torch.tensor([[0., 1.], [0., -1.]]))
    modele.bias.zero_()


### Choisir les paramètres et leur mise à jour

Le modèle et ses entrées se rejoignent avant le calcul.

In [7]:
modele = modele.to(device)
X = torch.tensor([[0.1, 1.5], [0.2, 4.5], [0.15, 2.5]]).to(device)
cibles = torch.tensor([0, 1, 0], dtype=torch.long).to(device)
optimiseur = torch.optim.SGD(modele.parameters(), lr=0.1)


In [8]:
observer('X', X)
observer('cibles', cibles)

X forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 1.5], [0.20000000298023224, 4.5], [0.15000000596046448, 2.5]]
cibles forme (3,) type torch.int64 requires_grad False
  valeurs : [0, 1, 0]


### Calculer l’état suivant

Le scaler est inactif sur CPU ; la copie des poids sert au contrôle du changement.

In [9]:
amp_active = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=amp_active)
poids_avant = modele.weight.detach().clone()


In [10]:
observer('amp_active', amp_active)
observer('scaler', scaler)
observer('poids_avant', poids_avant)

amp_active = False
poids_avant forme (2, 2) type torch.float32 requires_grad False
  valeurs : [[0.0, 1.0], [0.0, -1.0]]


### Remettre les gradients à zéro

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [11]:
optimiseur.zero_grad()


In [12]:
observer_entrainement({'device': device, 'modele': modele, 'X': X, 'cibles': cibles, 'optimiseur': optimiseur, 'amp_active': amp_active, 'scaler': scaler, 'poids_avant': poids_avant}, montrer_poids=False)

modele.weight.grad = None
modele.bias.grad = None


### Construire la règle de perte

Autocast couvre la passe avant, pas backward. Le scaler gère la remontée et l’utilisation des gradients.

In [13]:
with torch.amp.autocast(device_type=device.type, enabled=amp_active):
    scores = modele(X)
    perte = nn.CrossEntropyLoss()(scores, cibles)


### Calculer les gradients

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [14]:
scaler.scale(perte).backward()


In [15]:
observer_entrainement({'device': device, 'modele': modele, 'X': X, 'cibles': cibles, 'optimiseur': optimiseur, 'amp_active': amp_active, 'scaler': scaler, 'poids_avant': poids_avant}, montrer_poids=False)

modele.weight.grad forme (2, 2) type torch.float32 requires_grad False
  valeurs : [[0.06474293768405914, 1.470524787902832], [-0.06474293768405914, -1.4705246686935425]]
modele.bias.grad forme (2,) type torch.float32 requires_grad False
  valeurs : [0.3152526617050171, -0.3152526617050171]


### Appliquer une mise à jour

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [16]:
scaler.step(optimiseur)


In [17]:
observer_entrainement({'device': device, 'modele': modele, 'X': X, 'cibles': cibles, 'optimiseur': optimiseur, 'amp_active': amp_active, 'scaler': scaler, 'poids_avant': poids_avant}, montrer_poids=True)

modele.weight forme (2, 2) type torch.float32 requires_grad True
  valeurs : [[-0.006474293768405914, 0.8529475331306458], [0.006474293768405914, -0.8529475331306458]]
modele.weight.grad forme (2, 2) type torch.float32 requires_grad False
  valeurs : [[0.06474293768405914, 1.470524787902832], [-0.06474293768405914, -1.4705246686935425]]
modele.bias forme (2,) type torch.float32 requires_grad True
  valeurs : [-0.03152526542544365, 0.03152526542544365]
modele.bias.grad forme (2,) type torch.float32 requires_grad False
  valeurs : [0.3152526617050171, -0.3152526617050171]


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [18]:
scaler.update()


### Calculer l’état suivant

La somme parcourt seulement les valeurs des paramètres.

In [19]:
octets_poids = sum(p.numel() * p.element_size() for p in modele.parameters())


In [20]:
observer('octets_poids', octets_poids)

octets_poids = 24


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [21]:
print('device et AMP :', device.type, amp_active)
print('octets des parametres :', octets_poids)


device et AMP : cpu False
octets des parametres : 24


### Calculer l’état suivant

La synchronisation attend les travaux CUDA. Cette branche n’est pas exécutée sur le contrôle CPU.

In [22]:
if device.type == "cuda":
    torch.cuda.synchronize(device)
    print('octets alloues par PyTorch sur CUDA :', torch.cuda.memory_allocated(device))

In [23]:
assert bool(device.type == "cpu" and not amp_active and not scaler.is_enabled()), 'cuda-amp-memoire/main: device.type == "cpu" and not amp_active and not scaler.is_enabled()'
assert bool(octets_poids == 24), 'cuda-amp-memoire/main: octets_poids == 24'
assert bool(scores.shape == (3, 2) and cibles.dtype == torch.int64 and torch.isfinite(perte).item()), 'cuda-amp-memoire/main: scores.shape == (3, 2) and cibles.dtype == torch.int64 and torch.isfinite(perte).item()'
assert bool(not torch.equal(poids_avant, modele.weight)), 'cuda-amp-memoire/main: not torch.equal(poids_avant, modele.weight)'
assert bool(all(p.device == X.device == cibles.device for p in modele.parameters())), 'cuda-amp-memoire/main: all(p.device == X.device == cibles.device for p in modele.parameters())'
print("Contrôles numériques de la démonstration réussis.")

Contrôles numériques de la démonstration réussis.


Une branche matérielle et une mesure réelle de performance doivent rester distinguées.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Placez les entrées sur le même device que le modèle. Conservez leur type flottant.

In [24]:
exo_modele=torch.nn.Linear(2,2)
exo_device=next(exo_modele.parameters()).device
exo_entrees=torch.tensor([[.1,1.5],[.2,4.5]])
exo_placees = ...  # Remplacer les points par votre réponse.
if exo_placees is Ellipsis or exo_placees is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_placees.device == next(exo_modele.parameters()).device
    assert exo_placees.dtype == torch.float32
    torch.testing.assert_close(exo_placees,exo_entrees)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Affectez le tenseur renvoyé par to(device). Sur ce noyau, tout reste sur CPU ; aucune mesure CUDA n’est produite.

</details>

### Exercice 2

Combien d’octets logiques occupent les paramètres d’une Linear(2,2), avec biais, en float32 ?

In [25]:
exo_modele=torch.nn.Linear(2,2,dtype=torch.float32)
exo_octets = ...  # Remplacer les points par votre réponse.
if exo_octets is Ellipsis or exo_octets is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_octets == sum(p.numel()*p.element_size() for p in exo_modele.parameters())
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Quatre poids et deux biais font six valeurs de quatre octets. Les gradients et activations ne sont pas comptés.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_modele=torch.nn.Linear(2,2)
exo_device=next(exo_modele.parameters()).device
exo_entrees=torch.tensor([[.1,1.5],[.2,4.5]])
exo_placees = exo_entrees.to(exo_device)
if exo_placees is Ellipsis or exo_placees is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_placees.device == next(exo_modele.parameters()).device
    assert exo_placees.dtype == torch.float32
    torch.testing.assert_close(exo_placees,exo_entrees)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_modele=torch.nn.Linear(2,2,dtype=torch.float32)
exo_octets = 24
if exo_octets is Ellipsis or exo_octets is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_octets == sum(p.numel()*p.element_size() for p in exo_modele.parameters())
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [28]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()